# RSNA Knee Abnormality Detection — First Submission
**Team:** Gladiators (Utkarsh Milind Inamdar · Gaurav Hemant Lokhande)

**Pipeline:**
1. Re-generate pseudo-labels from radiology reports (NLP keyword matching)
2. Train EfficientNet-B0 on 3D MRI volumes (2D-slice aggregation)
3. Run inference on test set → `submission.csv`

In [ ]:
# Install timm if not already present (Kaggle usually has it)
!pip install -q timm
print('Setup done.')

In [ ]:
import os, re, unicodedata, sys
import numpy as np
import pandas as pd
import cv2
import pydicom
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy import ndimage
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import timm

print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────────
DATA_DIR        = '/kaggle/input/rsna-knee-abnormality-detection'
WORKING_DIR     = '/kaggle/working'

TRAIN_CSV       = f'{DATA_DIR}/train.csv'
TRAIN_SERIES    = f'{DATA_DIR}/train_series.csv'
TEST_CSV        = f'{DATA_DIR}/test.csv'
TEST_SERIES     = f'{DATA_DIR}/test_series.csv'
SAMPLE_SUB      = f'{DATA_DIR}/sample_submission.csv'
PSEUDO_CSV      = f'{WORKING_DIR}/train_pseudo_labeled.csv'
MODEL_PATH      = f'{WORKING_DIR}/model_best.pt'
SUBMISSION_PATH = f'{WORKING_DIR}/submission.csv'

PLANE         = 'Sagittal'
TARGET_SLICES = 24
IMG_SIZE      = 224
BACKBONE      = 'efficientnet_b0'
EPOCHS        = 10
BATCH_SIZE    = 8
LR            = 1e-4
SEED          = 42
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_WORKERS   = 2

LABEL_COLS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

os.makedirs(WORKING_DIR, exist_ok=True)
print(f'Device: {DEVICE} | Data: {DATA_DIR}')

## Stage 1 — Pseudo-Label Generation
Extracts labels from multilingual radiology reports using a keyword lexicon with negation detection.
Fills all 58,555 unlabeled studies; preserves the 58 clinician ground-truth labels.

In [ ]:
LEXICON = {
    'ACL': {'keywords': ['acl', 'lca', 'cruzado anterior', 'cruciate', 'capraz', 'kruisband', 'ruptur', 'ρήξη', 'χιαστού']},
    'MCL': {'keywords': ['mcl', 'lcm', 'colateral medial', 'collaterale', 'yan bag', 'innenband', 'πλαγίου']},
    'Medial Meniscus': {'keywords': ['medial meniscus', 'menisco medial', 'menisco interno', 'ic meniskus', 'mediale meniscus', 'innenmeniskus', 'ménisque médial', 'έσω μηνίσκου']},
    'Lateral Meniscus': {'keywords': ['lateral meniscus', 'menisco lateral', 'menisco externo', 'dis meniskus', 'laterale meniscus', 'außenmeniskus', 'ménisque latéral', 'έξω μηνίσκου']},
    'Medial OA': {'keywords': ['medial oa', 'artrosis medial', 'artrosis femorotibial medial', 'medial artroz', 'mediale artrose', 'mediale gonarthrose', 'arthrose médiale']},
    'Lateral OA': {'keywords': ['lateral oa', 'artrosis lateral', 'artrosis femorotibial lateral', 'dis artroz', 'laterale artrose', 'laterale gonarthrose', 'arthrose latérale']},
    'PF OA': {'keywords': ['pf oa', 'patellofemoral', 'patelofemoral', 'fémoro-patellaire', 'femorotibial lateral', 'patellofemorale', 'επιγoνατιδoμηριαίας']},
    'Effusion': {'keywords': ['effusion', 'derrame', 'efüzyon', 'kniegelenkerguss', 'épanchement', 'συλλογή']},
    'Synovitis': {'keywords': ['synovitis', 'sinovitis', 'sinovit', 'synovite', 'συνοβίτιδα']},
    "Baker's": {'keywords': ['baker', 'popliteal', 'quiste de baker', 'popliteal kist', 'poplitealzyste', 'kyste de baker', 'κύστη baker']},
    'Contusion': {'keywords': ['contusion', 'bruise', 'bruising', 'contusión', 'kemik kontüzyonu', 'contusie', 'kontusion', 'οίδημα']},
    'Fracture': {'keywords': ['fracture', 'fractura', 'kirik', 'fractuur', 'fraktur', 'κατάγμα']}
}

PRE_NEGATIONS  = r'\b(no|sin|not|negative for|without|free of|keine|kein|pas de|sans|normal|conservada|conservado|intacto|intacta)\b'
POST_NEGATIONS = r'\b(izlenmedi|saptanmadi|yok|not seen|absent|normaal|intact|normal)\b'

def clean_and_normalize(text):
    if not isinstance(text, str):
        return ''
    nfkd = unicodedata.normalize('NFKD', text)
    return ''.join(c for c in nfkd if not unicodedata.combining(c)).lower()

def check_negation(text, keyword, match_start, window=5):
    prefix_words = re.findall(r'\w+', text[max(0, match_start - 60):match_start])[-window:]
    if any(re.match(PRE_NEGATIONS, w) for w in prefix_words):
        return True
    suffix_words = re.findall(r'\w+', text[match_start + len(keyword):match_start + len(keyword) + 60])[:window]
    if any(re.match(POST_NEGATIONS, w) for w in suffix_words):
        return True
    return False

def parse_report(report_text):
    cleaned = clean_and_normalize(report_text)
    result = {}
    for label, info in LEXICON.items():
        found = 0
        for kw in info['keywords']:
            for match in re.finditer(re.escape(kw), cleaned):
                if not check_negation(cleaned, kw, match.start()):
                    found = 1
                    break
            if found:
                break
        result[label] = found
    return result

print('Pseudo-label functions defined.')

In [ ]:
# Run pseudo-labeling (takes ~1-2 min for 58k studies)
df = pd.read_csv(TRAIN_CSV)
print(f'Loaded train.csv: {len(df)} studies')

pseudo_df = df.copy()
unlabeled_mask = pseudo_df[LABEL_COLS].isnull().all(axis=1)
print(f'Unlabeled studies to fill: {unlabeled_mask.sum()}')

for idx in tqdm(pseudo_df[unlabeled_mask].index, desc='pseudo-labeling'):
    parsed = parse_report(pseudo_df.at[idx, 'Report'])
    for col in LABEL_COLS:
        pseudo_df.at[idx, col] = parsed[col]

pseudo_df.to_csv(PSEUDO_CSV, index=False)
print(f'Saved pseudo-labeled CSV to: {PSEUDO_CSV}')
print('\nLabel prevalence (% positive):')
print((pseudo_df[LABEL_COLS] == 1).mean().mul(100).round(1).to_string())

## Stage 2 — Dataset and Model

In [ ]:
def load_dicom_slice(filepath):
    dcm = pydicom.dcmread(filepath)
    pixel_array = dcm.pixel_array.astype(np.float32)
    slope = getattr(dcm, 'RescaleSlope', 1.0)
    intercept = getattr(dcm, 'RescaleIntercept', 0.0)
    pixel_array = pixel_array * slope + intercept
    denom = pixel_array.max() - pixel_array.min()
    if denom == 0:
        return np.zeros_like(pixel_array)
    return (pixel_array - pixel_array.min()) / denom

def load_and_resize_series(series_dir, target_slices=24, target_size=(224, 224)):
    dcm_files = [os.path.join(series_dir, f) for f in os.listdir(series_dir) if f.endswith('.dcm')]
    if not dcm_files:
        raise ValueError(f'No DICOM files in: {series_dir}')

    def get_z(fp):
        try:
            return float(pydicom.dcmread(fp, stop_before_pixels=True).ImagePositionPatient[2])
        except Exception:
            return float(os.path.splitext(os.path.basename(fp))[0])

    dcm_files.sort(key=get_z)
    volume = np.stack([cv2.resize(load_dicom_slice(f), target_size, interpolation=cv2.INTER_LINEAR) for f in dcm_files], axis=0)

    if volume.shape[0] != target_slices:
        volume = ndimage.zoom(volume, (target_slices / volume.shape[0], 1, 1), order=1)
    return volume


class RSNAKneeDataset(Dataset):
    def __init__(self, csv_file, series_csv, data_dir, plane='Sagittal',
                 target_slices=24, target_size=(224, 224), is_train=True):
        self.data_dir = data_dir
        self.target_slices = target_slices
        self.target_size = target_size
        self.is_train = is_train
        self.label_cols = LABEL_COLS

        self.labels_df = pd.read_csv(csv_file)
        series_df = pd.read_csv(series_csv)
        plane_series = series_df[series_df['Anatomical_Plane'].str.lower() == plane.lower()]
        self.plane_series = plane_series
        self.studies = plane_series['StudyInstanceUID'].unique()

    def __len__(self):
        return len(self.studies)

    def __getitem__(self, idx):
        study_uid = self.studies[idx]
        series_uid = self.plane_series[
            self.plane_series['StudyInstanceUID'] == study_uid
        ].iloc[0]['SeriesInstanceUID']

        subfolder = 'train_series' if self.is_train else 'test_series'
        series_dir = os.path.join(self.data_dir, subfolder, study_uid, series_uid)

        try:
            volume = load_and_resize_series(series_dir, self.target_slices, self.target_size)
        except Exception:
            volume = np.zeros((self.target_slices, *self.target_size), dtype=np.float32)

        volume_tensor = torch.tensor(volume, dtype=torch.float32).unsqueeze(0)

        study_labels = self.labels_df[self.labels_df['StudyInstanceUID'] == study_uid]
        if len(study_labels) > 0 and self.is_train:
            targets = study_labels.iloc[0][self.label_cols].values.astype(np.float32)
        else:
            targets = np.zeros(len(self.label_cols), dtype=np.float32)

        return volume_tensor, torch.tensor(targets, dtype=torch.float32)


class RSNAKneeModel(nn.Module):
    def __init__(self, backbone='efficientnet_b0', num_classes=12, pretrained=True):
        super().__init__()
        self.encoder = timm.create_model(backbone, pretrained=pretrained, in_chans=1, num_classes=0, global_pool='avg')
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(self.encoder.num_features, num_classes))

    def forward(self, x):
        B, C, D, H, W = x.shape
        x = x.squeeze(1).reshape(B * D, 1, H, W)
        feats = self.encoder(x).reshape(B, D, -1).mean(dim=1)
        return self.head(feats)

print('Dataset and Model classes defined.')

## Stage 3 — Training

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)

df = pd.read_csv(PSEUDO_CSV)
df = df.dropna(subset=LABEL_COLS, how='all').reset_index(drop=True)
df[LABEL_COLS] = df[LABEL_COLS].fillna(0).astype(float)

study_ids = df['StudyInstanceUID'].unique()
train_ids, val_ids = train_test_split(study_ids, test_size=0.2, random_state=SEED)

os.makedirs('/kaggle/working/tmp', exist_ok=True)
df[df['StudyInstanceUID'].isin(train_ids)].reset_index(drop=True).to_csv('/kaggle/working/tmp/train_split.csv', index=False)
df[df['StudyInstanceUID'].isin(val_ids)].reset_index(drop=True).to_csv('/kaggle/working/tmp/val_split.csv', index=False)

shared_kwargs = dict(series_csv=TRAIN_SERIES, data_dir=DATA_DIR, plane=PLANE,
                     target_slices=TARGET_SLICES, target_size=(IMG_SIZE, IMG_SIZE), is_train=True)

train_ds = RSNAKneeDataset(csv_file='/kaggle/working/tmp/train_split.csv', **shared_kwargs)
val_ds   = RSNAKneeDataset(csv_file='/kaggle/working/tmp/val_split.csv',   **shared_kwargs)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train studies: {len(train_ds)} | Val studies: {len(val_ds)}')

device = torch.device(DEVICE)
model = RSNAKneeModel(backbone=BACKBONE, pretrained=True).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    train_loss, steps = 0.0, 0
    for volumes, targets in tqdm(train_loader, desc=f'Epoch {epoch:02d} train', leave=False):
        volumes, targets = volumes.to(device), targets.to(device)
        logits = model(volumes)
        mask = ~targets.isnan()
        loss = criterion(logits[mask], targets[mask])
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        train_loss += loss.item(); steps += 1
    train_loss /= max(steps, 1)

    # --- Validate ---
    model.eval()
    val_loss, steps = 0.0, 0
    with torch.no_grad():
        for volumes, targets in tqdm(val_loader, desc=f'Epoch {epoch:02d} val', leave=False):
            volumes, targets = volumes.to(device), targets.to(device)
            logits = model(volumes)
            mask = ~targets.isnan()
            loss = criterion(logits[mask], targets[mask])
            val_loss += loss.item(); steps += 1
    val_loss /= max(steps, 1)

    scheduler.step()
    print(f'Epoch {epoch:02d}/{EPOCHS} | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), MODEL_PATH)
        print(f'  -> Saved best model (val_loss={val_loss:.4f})')

print(f'\nTraining done. Best val_loss: {best_val_loss:.4f}')

## Stage 4 — Inference & Submission

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print(f'Loaded best checkpoint from: {MODEL_PATH}')

test_ds = RSNAKneeDataset(
    csv_file=TEST_CSV, series_csv=TEST_SERIES, data_dir=DATA_DIR,
    plane=PLANE, target_slices=TARGET_SLICES, target_size=(IMG_SIZE, IMG_SIZE), is_train=False
)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(f'Test studies: {len(test_ds)}')

all_preds = []
with torch.no_grad():
    for volumes, _ in tqdm(test_loader, desc='inference'):
        logits = model(volumes.to(device))
        all_preds.append(torch.sigmoid(logits).cpu().numpy())

all_preds = np.concatenate(all_preds, axis=0)

# Build submission aligned with sample_submission.csv
series_df = pd.read_csv(TEST_SERIES)
plane_series = series_df[series_df['Anatomical_Plane'].str.lower() == PLANE.lower()]
ordered_uids = plane_series['StudyInstanceUID'].unique()

submission = pd.DataFrame(all_preds, columns=LABEL_COLS)
submission.insert(0, 'StudyInstanceUID', ordered_uids[:len(submission)])

sample_sub = pd.read_csv(SAMPLE_SUB)
submission = sample_sub[['StudyInstanceUID']].merge(submission, on='StudyInstanceUID', how='left')
submission[LABEL_COLS] = submission[LABEL_COLS].fillna(0.5)

submission.to_csv(SUBMISSION_PATH, index=False)
print(f'Submission saved: {SUBMISSION_PATH}')
print(f'Shape: {submission.shape}')
submission.head()